<a href="https://colab.research.google.com/github/SXXNHU/kwu-computer-network/blob/master/KW_MMDS_Colab_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# KW_MMDS - Colab 2
## Frequent Pattern Mining in Spark

### Setup

Let's setup Spark on your Colab environment.  Run the cell below!

In [1]:
!pip install pyspark
!pip install -U -q PyDrive
!apt install openjdk-8-jdk-headless -qq
import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-8-openjdk-amd64"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 987.4/987.4 kB 15.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
The following additional packages will be installed:
  libxtst6 openjdk-8-jre-headless
Suggested packages:
  openjdk-8-demo openjdk-8-source libnss-mdns fonts-dejavu-extra fonts-nanum
  fonts-ipafont-gothic fonts-ipafont-mincho fonts-wqy-microhei
  fonts-wqy-zenhei fonts-indic
The following NEW packages will be installed:
  libxtst6 openjdk-8-jdk-headless openjdk-8-jre-headless
0 upgraded, 3 newly installed, 0 to remove and 38 not upgraded.
Need to get 39.6 MB of archives.
After this operation, 144 MB of additional disk space will be used.
Selecting previously unselected package libxtst6:amd64.
(Reading database ... 126675 files and directories currently installed.)
Preparing to unpack .../libxtst6_2%3a1.2.3-1build4_amd64.deb ...
Unpacking libxtst6:amd64 (2:1.2.3-1build4) ...
Selecting previously unselected package openjdk-8-jre-headless:amd64.
Preparing to unpack .

Now we authenticate a Google Drive client to download the file we will be processing in our Spark job.

**Make sure to follow the interactive instructions.**

In [2]:
from pydrive2.auth import GoogleAuth
from pydrive2.drive import GoogleDrive
from google.colab import auth
from oauth2client.client import GoogleCredentials

# Authenticate and create the PyDrive client
auth.authenticate_user()
gauth = GoogleAuth()
gauth.credentials = GoogleCredentials.get_application_default()
drive = GoogleDrive(gauth)

In [3]:
id='1dhi1F78ssqR8gE6U-AgB80ZW7V_9snX4'
downloaded = drive.CreateFile({'id': id})
downloaded.GetContentFile('products.csv')

id='1KZBNEaIyMTcsRV817us6uLZgm-Mii8oU'
downloaded = drive.CreateFile({'id': id})
downloaded.GetContentFile('order_products__train.csv')

If you executed the cells above, you should be able to see the dataset we will need for this Colab under the "Files" tab on the left panel.

In [4]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

import pyspark
from pyspark.sql import *
from pyspark.sql.functions import *
from pyspark import SparkContext, SparkConf

Let's initialize the Spark context.

In [5]:
# create the session
conf = SparkConf().set("spark.ui.port", "4050")

# create the context
sc = pyspark.SparkContext(conf=conf)
spark = SparkSession.builder.getOrCreate()

### Your task

If you run successfully the setup stage, you are ready to work with the **3 Million Instacart Orders** dataset. In case you want to read more about it, check the [official Instacart blog post](https://tech.instacart.com/3-million-instacart-orders-open-sourced-d40d29ead6f2) about it, a concise [schema description](https://gist.github.com/jeremystan/c3b39d947d9b88b3ccff3147dbcf6c6b) of the dataset, and the [download page](https://www.instacart.com/datasets/grocery-shopping-2017).

In this Colab, we will be working only with a small training dataset (~131K orders) to perform fast Frequent Pattern Mining with the FP-Growth algorithm.

In [6]:
products = spark.read.csv('products.csv', header=True, inferSchema=True)
orders = spark.read.csv('order_products__train.csv', header=True, inferSchema=True)

In [7]:
products.printSchema()

root
 |-- product_id: integer (nullable = true)
 |-- product_name: string (nullable = true)
 |-- aisle_id: string (nullable = true)
 |-- department_id: string (nullable = true)



In [8]:
orders.printSchema()

root
 |-- order_id: integer (nullable = true)
 |-- product_id: integer (nullable = true)
 |-- add_to_cart_order: integer (nullable = true)
 |-- reordered: integer (nullable = true)



Use the Spark Dataframe API to join 'products' and 'orders', so that you will be able to see the product names in each transaction (and not only their ids).  Then, group by the orders by 'order_id' to obtain one row per basket (i.e., set of products purchased together by one customer).

In this Colab we will explore [MLlib](https://spark.apache.org/mllib/), Apache Spark's scalable machine learning library. Specifically, you can use its implementation of the [FP-Growth](https://spark.apache.org/docs/latest/ml-frequent-pattern-mining.html#fp-growth) algorithm to perform efficiently Frequent Pattern Mining in Spark.
Use the Python example in the documentation, and train a model with

```minSupport=0.01``` and ```minConfidence=0.5```



1. (25점) Compute how many frequent itemsets and association rules were generated by running FP-growth.
- 중요: num_freqItemsets1에는 빈발 항목집합의 개수를 저장하고, num_associationRules1에는 연관 규칙의 개수를 저장합니다.

In [26]:
# 주석 삭제 금지
# 1번 문제의 코드는 현재 셀에만 작성하세요. (products와 orders의 join도 여기에서 수행)
# 최종 결과는 num_freqItemsets1과 num_associationRules1에 기록합니다.
from pyspark.ml.fpm import FPGrowth

orders_joined = orders.join(products, on='product_id')
orders_joined = orders_joined.selectExpr('order_id', 'product_name')

df = orders_joined.groupBy('order_id').agg(collect_set('product_name').alias('items'))

fpGrowth = FPGrowth(itemsCol="items", minSupport=0.01, minConfidence=0.5)
model = fpGrowth.fit(df)

num_freqItemsets1 = model.freqItemsets.count()
num_associationRules1 = model.associationRules.count()

print(num_freqItemsets1, num_associationRules1)

120 0


2. (25점) What is the most frequent item?
- 중요: most_freqItem에는 가장 많이 등장한 product의 product_name을 string으로 기재합니다.

In [28]:
# 주석 삭제 금지
# 2번 문제의 코드는 현재 셀에만 작성하세요.
# 최종 결과는 most_freqItem에 string으로 저장합니다.

most_freqItem = model.freqItemsets.orderBy(desc('freq')).first()['items'][0]
print(most_freqItem)

Banana


3. (25점) Now retrain the FP-growth model changing only
```minsupport=0.001```
and compute how many frequent itemsets and association rules were generated.
- 중요: num_freqItemsets2에는 빈발 항목집합의 개수를 저장하고, num_associationRules2에는 연관 규칙의 개수를 저장합니다.


In [29]:
# 주석 삭제 금지
# 3번 문제의 코드는 현재 셀에만 작성하세요.
# 최종 결과는 num_freqItemsets2와 num_associationRules2에 기록합니다.

fpGrowth = FPGrowth(itemsCol="items", minSupport=0.001, minConfidence=0.5)
model = fpGrowth.fit(df)

num_freqItemsets2 = model.freqItemsets.count()
num_associationRules2 = model.associationRules.count()

print(num_freqItemsets2, num_associationRules2)

4444 11


4. (25점) Collect all the association rules of the above problem.
- 중요: 3번 문제에서 찾은 모든 연관 규칙을 associationRules에 **confidence의 내림차순**으로 저장합니다.
- 중요: type(associationRules)는 pyspark.sql.dataframe.DataFrame 입니다.
- 중요: associationRules.show(truncate=False)를 수행하면 아래와 같이 출력됩니다. (첫번째 행만 표시)

<img src = "https://drive.google.com/uc?id=10-qoZ60-MyX8SzJYcqXb7jAFgrNHPDvW">

In [31]:
# 주석 삭제 금지
# 4번 문제의 코드는 현재 셀에만 작성하세요.
# 최종 결과는 associationRules에 저장합니다.

associationRules = model.associationRules.orderBy(desc('confidence'))
associationRules.show(truncate=False)

+-----------------------------------------------------------------+------------------------+------------------+------------------+---------------------+
|antecedent                                                       |consequent              |confidence        |lift              |support              |
+-----------------------------------------------------------------+------------------------+------------------+------------------+---------------------+
|[Organic Raspberries, Organic Hass Avocado, Organic Strawberries]|[Bag of Organic Bananas]|0.5984251968503937|5.072272070642333 |0.0017376856770495927|
|[Organic Cucumber, Organic Hass Avocado, Organic Strawberries]   |[Bag of Organic Bananas]|0.546875          |4.635330870478036 |0.0010669999771357147|
|[Organic Kiwi, Organic Hass Avocado]                             |[Bag of Organic Bananas]|0.5459770114942529|4.627719489738336 |0.001448071397541327 |
|[Organic Navel Orange, Organic Raspberries]                      |[Bag of Organic

5. (0점) 연관 규칙 살펴보기

In [32]:
# 주석 삭제 금지
# 코드를 작성할 필요는 없습니다.
# 위에서 찾아낸 연관 규칙들을 종합하면 어떤 것을 알 수 있는지 생각해 보세요.
다양한 상품 조합의 결과로 반복해서 등장하는 Bag of Organic Bananas의 경우, 특정 상품의 구매 여부와 관계없이 구매 확률이 높아보입니다.
이는 Bag of Organic Bananas가 해당 플랫폼의 주력 상품이고, 소비자들이 이를 기본 과일로 인식하고 장바구니에 담고 있다고 생각합니다.
즉, 어떤 상품을 구매하던, organic 상품을 구매하고 있는 사용자에게 Bag of Organic Bananas 상품을 노출시키는 것은 유의미한 전략이라고 생각합니다.

덧붙여, 아마도 소비자들은 Bag of Organic Bananas을 단독으로 소비하기보다는, 다양한 유기농 제품들을 조합하여 건강식을 준비할 확률이 높다고 사료됩니다.
이러한 판단의 근거는 다양한 antecedent 조합을 통하여 유추하였습니다. 이를 통해 바나나가 이용되는 플래터,셰이크,스무디 등의 재료를 노출시키는 것도 좋은 전략이라고 생각합니다.




SyntaxError: invalid syntax (ipython-input-3387103605.py, line 4)